In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch02-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "02-05-conformers-and-reactions.py"

# 從正丁醇構象到醇氧化的候選結構

以實際 ESOL 結構進行操作，四個反應對照為概念示意。候選結構不代表實測產物。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import os, json
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolTransforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
_, parts, _ = dc.molnet.load_delaney(featurizer=dc.feat.RawFeaturizer(smiles=True), splitter='index', transformers=[], reload=False, data_dir=os.environ.get('DEEPCHEM_DATA_DIR'))
rows=[]
for part in parts:
 for smi in part.ids:
  mol=Chem.MolFromSmiles(smi)
  assert mol is not None
  rows.append((len(rows)+1,str(smi),mol))
def lookup(smiles):
 key=Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
 return next(row for row in rows if Chem.MolToSmiles(row[2])==key)
row_id, original, base=lookup('CCCCO')
mol=Chem.AddHs(base)
params=AllChem.ETKDGv3(); params.randomSeed=42; params.numThreads=1
ids=list(AllChem.EmbedMultipleConfs(mol,numConfs=12,params=params))
assert ids and AllChem.MMFFHasAllMoleculeParams(mol)
results=AllChem.MMFFOptimizeMoleculeConfs(mol,numThreads=1,maxIters=1000,mmffVariant='MMFF94')
converged=[(cid,float(energy)) for cid,(status,energy) in zip(ids,results) if status==0]
assert converged
emin=min(e for _,e in converged)
chain=mol.GetSubstructMatch(Chem.MolFromSmarts('[CH3]-[CH2]-[CH2]-[CH2]-[OH]'))
angles=[rdMolTransforms.GetDihedralDeg(mol.GetConformer(cid),*chain[:4]) for cid,_ in converged]
energies=[e-emin for _,e in converged]
print('Butan-1-ol ESOL row',row_id,'requested=12 embedded=',len(ids),'converged=',len(converged))
print('relative energies kcal/mol', [round(x,4) for x in energies])
# This rule enumerates a local structural change; it does not predict a reaction yield.
rxn=AllChem.ReactionFromSmarts('[C;X4;H1,H2;+0:1]-[O;X2;H1;+0:2]>>[C:1]=[O:2]')
def products_for(m):
 products={}
 for bundle in rxn.RunReactants((m,)):
  for product in bundle:
   Chem.SanitizeMol(product)
   products[Chem.MolToSmiles(product)]=product
 return products
# 概念示意 — 實際應用請使用下方 DeepChem 版本
entries=[]
for name,smi in [('Ethanol','CCO'),('Propan-2-ol','CC(O)C'),('tert-Butanol','CC(C)(C)O'),('Phenol','Oc1ccccc1')]:
 reactant=Chem.MolFromSmiles(smi)
 products=products_for(reactant)
 print(name,'concept unique candidates',list(products))
 entries.append((name,reactant,products))
assert list(entries[0][2])==['CC=O']
assert list(entries[1][2])==['CC(C)=O']
assert not entries[2][2] and not entries[3][2]
# DeepChem 版本：對每筆 ESOL 結構套用規則，保留原資料列編號。
matched=[]
for rid,raw,reactant in rows:
 candidates=products_for(reactant)
 if candidates:
  matched.append((rid,raw,sorted(candidates)))
print('ESOL matched rows',len(matched))
assert any(rid==984 and products==['CC=O'] for rid,raw,products in matched)
out=Path('../../figures') if Path('../../_quarto.yml').is_file() else Path('figures')
out.mkdir(exist_ok=True)
fig=plt.figure(figsize=(11,4.6))
ax=fig.add_subplot(121,projection='3d')
best=min(converged,key=lambda pair:pair[1])[0];conf=mol.GetConformer(best)
for bond in base.GetBonds():
 a,b=(conf.GetAtomPosition(i) for i in (bond.GetBeginAtomIdx(),bond.GetEndAtomIdx()))
 ax.plot([a.x,b.x],[a.y,b.y],[a.z,b.z],color='#64748b',lw=3)
for atom in base.GetAtoms():
 p=conf.GetAtomPosition(atom.GetIdx());ax.scatter(p.x,p.y,p.z,color=('#dc2626' if atom.GetSymbol()=='O' else '#334155'),s=75)
 ax.text(p.x,p.y,p.z,atom.GetSymbol())
ax.set(xlabel='x (Å)',ylabel='y (Å)',zlabel='z (Å)',title='Butan-1-ol: one generated conformer')
coords=conf.GetPositions()[:base.GetNumAtoms()]
center=(coords.max(axis=0)+coords.min(axis=0))/2
radius=float((coords.max(axis=0)-coords.min(axis=0)).max())/2+0.3
ax.set_xlim(center[0]-radius,center[0]+radius)
ax.set_ylim(center[1]-radius,center[1]+radius)
ax.set_zlim(center[2]-radius,center[2]+radius)
ax.set_box_aspect((1,1,1))
ax2=fig.add_subplot(122)
ax2.scatter(angles,energies,color='#0f766e',s=50)
ax2.set(xlabel='C–C–C–C dihedral (degrees)',ylabel='Relative MMFF94 energy (kcal/mol)',title='12 generated starting conformers')
ax2.grid(alpha=.2);fig.tight_layout();fig.savefig(out/'02-05-conformers-and-reactions.png',dpi=160);plt.close(fig)
draw_mols=[];legends=[]
for name,reactant,products in entries:
 draw_mols.extend([reactant,next(iter(products.values()),None)])
 legends.extend([name,'Candidate' if products else 'No match for this rule'])
panel=Draw.MolsToGridImage(draw_mols,molsPerRow=2,subImgSize=(350,180),legends=legends)
from PIL import ImageDraw, ImageFont
canvas=ImageDraw.Draw(panel)
font=ImageFont.load_default(size=18)
for row,(_,_,products) in enumerate(entries):
 if not products:
  canvas.text((525,row*180+155),'No match for this rule',font=font,fill='black',anchor='mm')
panel.save(out/'02-05-reaction-candidates.png')


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "02-05-conformers-and-reactions.png")))
display(Image(filename=str(figure_dir / "02-05-reaction-candidates.png")))
